# Bloque de codificación tipo Hamming

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from time import perf_counter
#Prueba de que el algoritmo funciona
# Función para codificar un nibble (4 bits de datos)

def h74_encode_nibble(nib):
    """Codificador Hamming (7,4) — solo codifica (sin decodificar)."""
    b3 = (nib >> 3) & 1
    b2 = (nib >> 2) & 1
    b1 = (nib >> 1) & 1
    b0 = (nib >> 0) & 1

    c = np.zeros(7, dtype=np.uint8)
    # Datos
    c[2], c[4], c[5], c[6] = b3, b2, b1, b0
    # Paridades (posiciones 1,2,4)
    c[0] = c[2] ^ c[4] ^ c[6]  # p1 cubre 1,3,5,7
    c[1] = c[2] ^ c[5] ^ c[6]  # p2 cubre 2,3,6,7
    c[3] = c[4] ^ c[5] ^ c[6]  # p4 cubre 4,5,6,7
    return c

def bytes_to_h74_bitstream(data):
    """Convierte lista de enteros (0–255) a bitstream Hamming(7,4)."""
    out = []
    for byte in data:
        hi = (byte >> 4) & 0xF
        lo = byte & 0xF
        out.extend(h74_encode_nibble(hi))
        out.extend(h74_encode_nibble(lo))
    return np.array(out, dtype=np.uint8)


def medir_tiempo(data, repeticiones=200):
    """Mide tiempo promedio de codificación Hamming(7,4)."""
    t0 = perf_counter()
    for _ in range(repeticiones):
        _ = bytes_to_h74_bitstream(data)
    t1 = perf_counter()
    print(f"Tiempo promedio: {(t1 - t0)/repeticiones*1e3:.4f} ms por corrida")


bpm_filtrado = [72, 75, 80, 90, 95]  # datos de ejemplo
print("Datos de entrada:", bpm_filtrado)

bits_codificados = bytes_to_h74_bitstream(bpm_filtrado)
print("Bits codificados (primeros 28 bits):", bits_codificados[:28], "...")
print("Tamaño total del bitstream:", len(bits_codificados), "bits")

# Medición de tiempo
medir_tiempo(bpm_filtrado)

Datos de entrada: [72, 75, 80, 90, 95]
Bits codificados (primeros 28 bits): [1 0 0 1 1 0 0 1 1 1 0 0 0 0 1 0 0 1 1 0 0 0 1 1 0 0 1 1] ...
Tamaño total del bitstream: 70 bits
Tiempo promedio: 0.0426 ms por corrida


In [2]:
def h74_encode_nibble(nib:int):
    b3 = (nib >> 3) & 1; b2 = (nib >> 2) & 1; b1 = (nib >> 1) & 1; b0 = (nib >> 0) & 1 #separa los 4 bits del nibble
    c = [0]*7 #lista vacía donde se guardará la palabra hamming
    c[2], c[4], c[5], c[6] = b3, b2, b1, b0 # asignación de posiciones
    c[0] = c[2] ^ c[4] ^ c[6] #cálculo de los bits de paridad
    c[1] = c[2] ^ c[5] ^ c[6]
    c[3] = c[4] ^ c[5] ^ c[6]
    return c #devuelve los bits codificados

def bytes_to_bits_h74(u8_list):
    out=[]
    for b in u8_list:
        hi=(b>>4)&0xF; lo=b&0xF #divide cada byte en dos nibbles
        out += h74_encode_nibble(hi) #aplica el codificador hamming a cada nibble
        out += h74_encode_nibble(lo)
    return np.array(out, dtype=np.uint8) #concatena los resultados

def verificar_h74(bits):
    assert len(bits) % 7 == 0 #verifica que los bloques sean múltiplos de 7
    malos=0  #sumador de bloques malos
    for i in range(0,len(bits),7):
        c=bits[i:i+7] #recorre los bits de 7 en 7 recalcula paridades (síndrome)
        s1=c[0]^c[2]^c[4]^c[6]
        s2=c[1]^c[2]^c[5]^c[6]
        s4=c[3]^c[4]^c[5]^c[6]
        if (s1|s2|s4)!=0: malos+=1
    return (len(bits)//7, malos)

def tiempo_stats_h74(data_bytes, repeticiones=200):
    ts=[]
    for _ in range(repeticiones):
        t0=perf_counter(); _=bytes_to_bits_h74(data_bytes); t1=perf_counter()
        ts.append(t1-t0)
    ts=np.array(ts); return ts.mean()
